# Treinamento — Sistema de Recomendação de Filmes

Este notebook executa os experimentos do projeto:

1. **Popularidade**
2. **SVD truncado**
3. **Matrix Factorization com embeddings usando PyTorch**

A ideia é usar os arquivos do MovieLens em:

```text
data/ml_data/
```

Arquivos esperados:

```text
ratings.csv
movies.csv
```

O notebook foi escrito para usar principalmente:

```text
pandas
numpy
torch
```

Não é necessário `scikit-learn` para estes experimentos.

## 1. Configuração inicial

Nesta parte vamos:

- importar as bibliotecas;
- encontrar a pasta do projeto;
- localizar os arquivos `ratings.csv` e `movies.csv`;
- criar pastas para salvar os resultados.

O caminho foi feito de forma flexível para funcionar tanto se o notebook for aberto dentro da pasta `notebooks/` quanto se for executado pela raiz do projeto.

In [1]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def find_project_root():
    """Tenta encontrar a raiz do projeto procurando por data/ml_data."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "ml_data").exists():
            return candidate
    # fallback comum: notebook dentro de notebooks/
    if Path.cwd().name == "notebooks":
        return Path.cwd().parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "ml_data"
RESULTS_DIR = PROJECT_ROOT / "results"
IMAGES_DIR = PROJECT_ROOT / "docs" / "images"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", PROJECT_ROOT)
print("Pasta de dados:", DATA_DIR)
print("Pasta de resultados:", RESULTS_DIR)
print("Pasta de imagens:", IMAGES_DIR)

ratings_files = list(DATA_DIR.rglob("ratings.csv"))
movies_files = list(DATA_DIR.rglob("movies.csv"))

if not ratings_files:
    raise FileNotFoundError(f"Não encontrei ratings.csv dentro de {DATA_DIR}")
if not movies_files:
    raise FileNotFoundError(f"Não encontrei movies.csv dentro de {DATA_DIR}")

RATINGS_PATH = ratings_files[0]
MOVIES_PATH = movies_files[0]

print("ratings.csv:", RATINGS_PATH)
print("movies.csv:", MOVIES_PATH)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo PyTorch:", DEVICE)

Raiz do projeto: /Users/eduardo/Desktop/Insper/projeto_recommendation_system
Pasta de dados: /Users/eduardo/Desktop/Insper/projeto_recommendation_system/data/ml_data
Pasta de resultados: /Users/eduardo/Desktop/Insper/projeto_recommendation_system/results
Pasta de imagens: /Users/eduardo/Desktop/Insper/projeto_recommendation_system/docs/images
ratings.csv: /Users/eduardo/Desktop/Insper/projeto_recommendation_system/data/ml_data/ratings.csv
movies.csv: /Users/eduardo/Desktop/Insper/projeto_recommendation_system/data/ml_data/movies.csv
Dispositivo PyTorch: cpu


## 2. Carregar os dados

Aqui carregamos:

- `ratings.csv`: avaliações dos usuários;
- `movies.csv`: nomes e gêneros dos filmes.

Também vamos olhar algumas estatísticas básicas para entender o tamanho do problema.

In [2]:
ratings = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)

print("ratings shape:", ratings.shape)
print("movies shape:", movies.shape)

display(ratings.head())
display(movies.head())

n_users_original = ratings["userId"].nunique()
n_movies_original = ratings["movieId"].nunique()
n_ratings = len(ratings)
rating_mean = ratings["rating"].mean()
rating_min = ratings["rating"].min()
rating_max = ratings["rating"].max()

print("Quantidade de usuários:", n_users_original)
print("Quantidade de filmes avaliados:", n_movies_original)
print("Quantidade de avaliações:", n_ratings)
print("Média das notas:", round(rating_mean, 4))
print("Menor nota:", rating_min)
print("Maior nota:", rating_max)

# Tabela simples da distribuição das notas.
rating_distribution = ratings["rating"].value_counts().sort_index().reset_index()
rating_distribution.columns = ["rating", "quantidade"]
display(rating_distribution)

rating_distribution.to_csv(RESULTS_DIR / "distribuicao_notas.csv", index=False)

ratings shape: (100836, 4)
movies shape: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Quantidade de usuários: 610
Quantidade de filmes avaliados: 9724
Quantidade de avaliações: 100836
Média das notas: 3.5016
Menor nota: 0.5
Maior nota: 5.0


,rating,quantidade
0,0.5,1370
1,1.0,2811
2,1.5,1791
3,2.0,7551
4,2.5,5550
5,3.0,20047
6,3.5,13136
7,4.0,26818
8,4.5,8551
9,5.0,13211


## 3. Preparar os identificadores

Os `userId` e `movieId` originais do MovieLens não começam necessariamente em zero e podem ter saltos.

Para trabalhar com matrizes e embeddings, vamos criar índices internos:

```text
user_idx: 0, 1, 2, ...
movie_idx: 0, 1, 2, ...
```

Isso facilita o uso com `numpy` e com `torch.nn.Embedding`.

In [3]:
user_ids = np.sort(ratings["userId"].unique())
movie_ids = np.sort(ratings["movieId"].unique())

user2idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie2idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
idx2user = {idx: user_id for user_id, idx in user2idx.items()}
idx2movie = {idx: movie_id for movie_id, idx in movie2idx.items()}

ratings["user_idx"] = ratings["userId"].map(user2idx).astype(np.int64)
ratings["movie_idx"] = ratings["movieId"].map(movie2idx).astype(np.int64)

n_users = len(user_ids)
n_movies = len(movie_ids)

movie_info = movies.set_index("movieId")

print("Usuários codificados:", n_users)
print("Filmes codificados:", n_movies)

display(ratings.head())

Usuários codificados: 610
Filmes codificados: 9724


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1,4.0,964982703,0,0
1,1,3,4.0,964981247,0,2
2,1,6,4.0,964982224,0,5
3,1,47,5.0,964983815,0,43
4,1,50,5.0,964982931,0,46


## 4. Separar treino e teste

Vamos separar as avaliações em:

- **treino**: usado para os modelos aprenderem;
- **teste**: usado apenas para medir se os modelos acertam avaliações que ficaram escondidas.

A separação será feita por usuário. Para cada usuário, aproximadamente 20% das avaliações ficam no teste.

Isso evita um problema: se fizéssemos uma divisão totalmente aleatória, poderia acontecer de algum usuário ficar sem avaliações no treino.

In [4]:
def train_test_split_by_user(df, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    test_indices = []

    for _, group in df.groupby("user_idx"):
        indices = group.index.to_numpy()
        if len(indices) <= 1:
            continue
        n_test = max(1, int(round(len(indices) * test_ratio)))
        n_test = min(n_test, len(indices) - 1)
        chosen = rng.choice(indices, size=n_test, replace=False)
        test_indices.extend(chosen.tolist())

    test_indices = set(test_indices)
    test_mask = df.index.isin(test_indices)

    train_df = df.loc[~test_mask].copy().reset_index(drop=True)
    test_df = df.loc[test_mask].copy().reset_index(drop=True)
    return train_df, test_df

train_df, test_df = train_test_split_by_user(ratings, test_ratio=0.2, seed=SEED)

print("Treino:", train_df.shape)
print("Teste:", test_df.shape)
print("Percentual teste:", round(len(test_df) / len(ratings), 4))

GLOBAL_MEAN = train_df["rating"].mean()
print("Média global do treino:", round(GLOBAL_MEAN, 4))

train_df.to_csv(RESULTS_DIR / "train_split.csv", index=False)
test_df.to_csv(RESULTS_DIR / "test_split.csv", index=False)

Treino: (80672, 6)
Teste: (20164, 6)
Percentual teste: 0.2
Média global do treino: 3.5034


## 5. Métricas de avaliação

Vamos medir dois tipos de resultado.

### Erro de previsão de nota

Usaremos:

$
RMSE = \sqrt{\frac{1}{|T|}\sum(r_{ui} - \hat{r}_{ui})^2}
$

$
MAE = \frac{1}{|T|}\sum |r_{ui} - \hat{r}_{ui}|
$

Quanto menor, melhor.

### Qualidade da recomendação Top-10

Um filme será considerado relevante quando a nota real no teste for maior ou igual a 4.

Vamos calcular:

- `Precision@10`
- `Recall@10`

In [5]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))


def clip_ratings(values, min_rating=0.5, max_rating=5.0):
    return np.clip(values, min_rating, max_rating)


def rating_metrics_from_predictions(name, y_true, y_pred):
    y_pred = clip_ratings(y_pred)
    return {
        "modelo": name,
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
    }

# Filmes vistos no treino por usuário.
train_seen = train_df.groupby("user_idx")["movie_idx"].apply(set).to_dict()

# Filmes relevantes no teste por usuário.
# Neste projeto: relevante = nota >= 4.
test_relevant = (
    test_df[test_df["rating"] >= 4.0]
    .groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)


def precision_recall_at_k(score_matrix, k=10):
    """Calcula Precision@K e Recall@K usando uma matriz usuário-filme de scores."""
    precisions = []
    recalls = []

    for user_idx, relevant_items in test_relevant.items():
        if len(relevant_items) == 0:
            continue

        scores = np.array(score_matrix[user_idx], copy=True)

        # Não recomendar filmes que o usuário já avaliou no treino.
        seen_items = train_seen.get(user_idx, set())
        if seen_items:
            scores[list(seen_items)] = -np.inf

        top_k = np.argpartition(-scores, kth=min(k, len(scores) - 1))[:k]
        top_k = top_k[np.argsort(-scores[top_k])]
        top_k_set = set(top_k.tolist())

        hits = len(top_k_set.intersection(relevant_items))
        precisions.append(hits / k)
        recalls.append(hits / len(relevant_items))

    return {
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        "usuarios_avaliados_topk": len(precisions),
    }

print("Funções de avaliação prontas.")
print("Usuários com pelo menos 1 filme relevante no teste:", len(test_relevant))

Funções de avaliação prontas.
Usuários com pelo menos 1 filme relevante no teste: 602


## 6. Modelo 1 — Popularidade

Este modelo recomenda filmes pela média de avaliação no treino.

Para evitar que um filme com apenas uma avaliação nota 5 fique no topo facilmente, usaremos uma média suavizada:

\[
score(i) = \frac{soma\_notas_i + m \cdot \mu}{quantidade_i + m}
\]

Onde:

- \(\mu\) é a média global;
- \(m\) é uma constante de suavização;
- filmes com poucas avaliações ficam mais próximos da média global.

In [6]:
def train_popularity(train_df, n_movies, global_mean, smoothing=10):
    stats = train_df.groupby("movie_idx")["rating"].agg(["sum", "count"])
    scores = np.full(n_movies, global_mean, dtype=np.float32)

    smoothed = (stats["sum"] + smoothing * global_mean) / (stats["count"] + smoothing)
    scores[stats.index.to_numpy()] = smoothed.to_numpy(dtype=np.float32)
    return scores

popularity_scores = train_popularity(train_df, n_movies, GLOBAL_MEAN, smoothing=10)

pop_test_pred = popularity_scores[test_df["movie_idx"].to_numpy()]
pop_rating_metrics = rating_metrics_from_predictions(
    "Popularidade",
    test_df["rating"].to_numpy(),
    pop_test_pred,
)

# Para Top-10, o mesmo vetor de popularidade é usado para todos os usuários.
pop_score_matrix = np.broadcast_to(popularity_scores, (n_users, n_movies))
pop_topk_metrics = precision_recall_at_k(pop_score_matrix, k=10)

print(pop_rating_metrics)
print(pop_topk_metrics)

{'modelo': 'Popularidade', 'RMSE': 0.9666860044585095, 'MAE': 0.7524666946763393}
{'Precision@10': 0.07790697674418605, 'Recall@10': 0.06136983853173653, 'usuarios_avaliados_topk': 602}


## 7. Modelo 2 — SVD truncado

O SVD decompõe a matriz de avaliações assim:

\[
R \approx U_k \Sigma_k V_k^T
\]

Como a matriz usuário-filme tem valores faltantes, vamos usar a seguinte estratégia:

1. Criar uma matriz com as notas do treino;
2. Calcular a média de cada usuário;
3. Subtrair a média do usuário nas notas conhecidas;
4. Preencher os valores faltantes com zero;
5. Aplicar SVD;
6. Reconstruir a matriz;
7. Somar a média do usuário de volta.

Assim, o zero representa aproximadamente: "sem desvio em relação à média do usuário".

In [7]:
def build_train_matrix(train_df, n_users, n_movies):
    matrix = np.full((n_users, n_movies), np.nan, dtype=np.float32)
    matrix[
        train_df["user_idx"].to_numpy(),
        train_df["movie_idx"].to_numpy(),
    ] = train_df["rating"].to_numpy(dtype=np.float32)
    return matrix

R_train = build_train_matrix(train_df, n_users, n_movies)

user_means = np.nanmean(R_train, axis=1)
user_means = np.where(np.isnan(user_means), GLOBAL_MEAN, user_means).astype(np.float32)

R_centered = np.where(np.isnan(R_train), 0.0, R_train - user_means[:, None]).astype(np.float32)

print("Matriz de treino:", R_train.shape)
print("Matriz centralizada:", R_centered.shape)
print("Calculando SVD. Isso pode levar alguns segundos...")

start = time.time()
U, S, Vt = np.linalg.svd(R_centered, full_matrices=False)
print("SVD concluído em segundos:", round(time.time() - start, 2))
print("U:", U.shape, "S:", S.shape, "Vt:", Vt.shape)


def svd_score_matrix(k):
    reconstructed_centered = (U[:, :k] * S[:k]) @ Vt[:k, :]
    scores = reconstructed_centered + user_means[:, None]
    return clip_ratings(scores).astype(np.float32)


def evaluate_svd_for_k(k):
    scores = svd_score_matrix(k)
    test_pred = scores[test_df["user_idx"].to_numpy(), test_df["movie_idx"].to_numpy()]
    rating_m = rating_metrics_from_predictions(f"SVD k={k}", test_df["rating"].to_numpy(), test_pred)
    topk_m = precision_recall_at_k(scores, k=10)
    return rating_m, topk_m, scores

K_VALUES = [5, 10, 20, 50, 100]
svd_k_rows = []
best_svd = {"k": None, "rmse": float("inf"), "scores": None}

for k in K_VALUES:
    if k > len(S):
        continue
    rating_m, topk_m, scores = evaluate_svd_for_k(k)
    row = {"k": k, "RMSE_SVD": rating_m["RMSE"], "MAE_SVD": rating_m["MAE"], **topk_m}
    svd_k_rows.append(row)
    print(row)

    if rating_m["RMSE"] < best_svd["rmse"]:
        best_svd = {"k": k, "rmse": rating_m["RMSE"], "scores": scores}

svd_k_results = pd.DataFrame(svd_k_rows)
display(svd_k_results)

svd_k_results.to_csv(RESULTS_DIR / "svd_k_results.csv", index=False)
print("Melhor k do SVD por RMSE:", best_svd["k"])

Matriz de treino: (610, 9724)
Matriz centralizada: (610, 9724)
Calculando SVD. Isso pode levar alguns segundos...
SVD concluído em segundos: 0.26
U: (610, 610) S: (610,) Vt: (610, 9724)
{'k': 5, 'RMSE_SVD': 0.9157036322080868, 'MAE_SVD': 0.7089145080280833, 'Precision@10': 0.11960132890365449, 'Recall@10': 0.09002064952908091, 'usuarios_avaliados_topk': 602}
{'k': 10, 'RMSE_SVD': 0.9140338713788471, 'MAE_SVD': 0.7075862816297164, 'Precision@10': 0.1270764119601329, 'Recall@10': 0.09715516803374537, 'usuarios_avaliados_topk': 602}
{'k': 20, 'RMSE_SVD': 0.9173982338708963, 'MAE_SVD': 0.7098691563442145, 'Precision@10': 0.12076411960132888, 'Recall@10': 0.09557900506927927, 'usuarios_avaliados_topk': 602}
{'k': 50, 'RMSE_SVD': 0.9258101190075054, 'MAE_SVD': 0.7172969718687451, 'Precision@10': 0.11063122923588041, 'Recall@10': 0.1076849269636564, 'usuarios_avaliados_topk': 602}
{'k': 100, 'RMSE_SVD': 0.9341682671243166, 'MAE_SVD': 0.7260710315213887, 'Precision@10': 0.08172757475083058, 'R

,k,RMSE_SVD,MAE_SVD,Precision@10,Recall@10,usuarios_avaliados_topk
0,5,0.915704,0.708915,0.119601,0.090021,602
1,10,0.914034,0.707586,0.127076,0.097155,602
2,20,0.917398,0.709869,0.120764,0.095579,602
3,50,0.925810,0.717297,0.110631,0.107685,602
4,100,0.934168,0.726071,0.081728,0.088550,602


Melhor k do SVD por RMSE: 10


## 8. Modelo 3 — Matrix Factorization com embeddings usando PyTorch

Este é o modelo principal.

A previsão da nota será:

\[
\hat{r}_{ui} = \mu + b_u + b_i + p_u^T q_i
\]

Onde:

- \(\mu\) é a média global;
- \(b_u\) é o ajuste do usuário;
- \(b_i\) é o ajuste do filme;
- \(p_u\) é o embedding do usuário;
- \(q_i\) é o embedding do filme.

O PyTorch vai aprender os embeddings automaticamente durante o treinamento.

In [8]:
class MatrixFactorizationModel(nn.Module):
    def __init__(self, n_users, n_movies, n_factors, global_mean):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, n_factors)
        self.movie_embedding = nn.Embedding(n_movies, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        self.register_buffer("global_mean", torch.tensor(float(global_mean), dtype=torch.float32))

        # Inicialização pequena para começar próximo da média global.
        nn.init.normal_(self.user_embedding.weight, mean=0.0, std=0.05)
        nn.init.normal_(self.movie_embedding.weight, mean=0.0, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, user_idx, movie_idx):
        user_vec = self.user_embedding(user_idx)
        movie_vec = self.movie_embedding(movie_idx)
        dot = (user_vec * movie_vec).sum(dim=1)
        user_b = self.user_bias(user_idx).squeeze(1)
        movie_b = self.movie_bias(movie_idx).squeeze(1)
        return self.global_mean + user_b + movie_b + dot


def make_dataloader(df, batch_size=4096, shuffle=True):
    users = torch.tensor(df["user_idx"].to_numpy(), dtype=torch.long)
    movies_t = torch.tensor(df["movie_idx"].to_numpy(), dtype=torch.long)
    ratings_t = torch.tensor(df["rating"].to_numpy(), dtype=torch.float32)
    dataset = TensorDataset(users, movies_t, ratings_t)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def evaluate_mf_rating_metrics(model, df):
    model.eval()
    users = torch.tensor(df["user_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
    movies_t = torch.tensor(df["movie_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        preds = model(users, movies_t).detach().cpu().numpy()
    preds = clip_ratings(preds)
    return rating_metrics_from_predictions("Matrix Factorization", df["rating"].to_numpy(), preds)


def mf_score_matrix(model, batch_users=128):
    """Gera matriz completa usuário-filme com as notas previstas pelo modelo."""
    model.eval()
    all_scores = []
    movie_indices = torch.arange(n_movies, dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        for start_idx in range(0, n_users, batch_users):
            end_idx = min(start_idx + batch_users, n_users)
            user_indices = torch.arange(start_idx, end_idx, dtype=torch.long, device=DEVICE)

            # Fórmula vetorizada: média + bias usuário + bias filme + embedding_user @ embedding_filme.T
            user_vecs = model.user_embedding(user_indices)
            movie_vecs = model.movie_embedding(movie_indices)
            user_b = model.user_bias(user_indices)
            movie_b = model.movie_bias(movie_indices).squeeze(1)

            scores = model.global_mean + user_b + movie_b.unsqueeze(0) + user_vecs @ movie_vecs.T
            all_scores.append(scores.detach().cpu().numpy())

    return clip_ratings(np.vstack(all_scores)).astype(np.float32)


def train_mf(n_factors=20, epochs=10, lr=0.01, weight_decay=1e-5, batch_size=4096):
    model = MatrixFactorizationModel(n_users, n_movies, n_factors, GLOBAL_MEAN).to(DEVICE)
    loader = make_dataloader(train_df, batch_size=batch_size, shuffle=True)

    # Adam é usado aqui por ser estável e simples para embeddings.
    # Ele ainda é um método de gradiente: ajusta os parâmetros para reduzir o erro.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_count = 0

        for batch_users, batch_movies, batch_ratings in loader:
            batch_users = batch_users.to(DEVICE)
            batch_movies = batch_movies.to(DEVICE)
            batch_ratings = batch_ratings.to(DEVICE)

            optimizer.zero_grad()
            preds = model(batch_users, batch_movies)
            loss = loss_fn(preds, batch_ratings)
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item()) * len(batch_ratings)
            total_count += len(batch_ratings)

        train_mse = total_loss / total_count
        test_metrics = evaluate_mf_rating_metrics(model, test_df)

        row = {
            "epoch": epoch,
            "k": n_factors,
            "train_mse": train_mse,
            "train_rmse": math.sqrt(train_mse),
            "test_rmse": test_metrics["RMSE"],
            "test_mae": test_metrics["MAE"],
        }
        history.append(row)
        print(row)

    return model, pd.DataFrame(history)

print("Classe e funções do modelo MF prontas.")

Classe e funções do modelo MF prontas.


## 9. Experimento 3 — Testar diferentes valores de `k`

Agora vamos testar diferentes quantidades de fatores latentes.

Valores planejados:

```text
k = 5, 10, 20, 50, 100
```

A ideia é verificar se aumentar a quantidade de fatores melhora ou piora o resultado.

Observação: se estiver demorando muito no seu computador, reduza `EPOCHS_BY_K` para 5 ou teste menos valores de `k`.

In [9]:
K_VALUES_MF = [5, 10, 20, 50, 100]
EPOCHS_BY_K = 10
LR = 0.01
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 4096

mf_k_rows = []
best_mf = {"k": None, "rmse": float("inf"), "model": None, "scores": None, "history": None}

for k in K_VALUES_MF:
    print("\n" + "=" * 80)
    print(f"Treinando Matrix Factorization com k={k}")
    print("=" * 80)

    model, history = train_mf(
        n_factors=k,
        epochs=EPOCHS_BY_K,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        batch_size=BATCH_SIZE,
    )

    final_metrics = evaluate_mf_rating_metrics(model, test_df)
    scores = mf_score_matrix(model)
    topk = precision_recall_at_k(scores, k=10)

    row = {
        "k": k,
        "RMSE_MF": final_metrics["RMSE"],
        "MAE_MF": final_metrics["MAE"],
        **topk,
    }
    mf_k_rows.append(row)
    print("Resultado final k=", k, row)

    history.to_csv(RESULTS_DIR / f"mf_training_history_k_{k}.csv", index=False)

    if final_metrics["RMSE"] < best_mf["rmse"]:
        best_mf = {
            "k": k,
            "rmse": final_metrics["RMSE"],
            "model": model,
            "scores": scores,
            "history": history,
        }

mf_k_results = pd.DataFrame(mf_k_rows)
display(mf_k_results)

mf_k_results.to_csv(RESULTS_DIR / "mf_k_results.csv", index=False)

if best_mf["model"] is not None:
    torch.save(best_mf["model"].state_dict(), RESULTS_DIR / "best_mf_model.pt")

print("Melhor k da Matrix Factorization por RMSE:", best_mf["k"])


Treinando Matrix Factorization com k=5
{'epoch': 1, 'k': 5, 'train_mse': 1.0132830637118875, 'train_rmse': 1.0066196221571917, 'test_rmse': 0.9726209344502966, 'test_mae': 0.7689195444389507}
{'epoch': 2, 'k': 5, 'train_mse': 0.8664551061658999, 'train_rmse': 0.9308357031001228, 'test_rmse': 0.9260343775170531, 'test_mae': 0.7188825276346817}
{'epoch': 3, 'k': 5, 'train_mse': 0.753095611533869, 'train_rmse': 0.8678108155202198, 'test_rmse': 0.8964606779303277, 'test_mae': 0.6912079400023967}
{'epoch': 4, 'k': 5, 'train_mse': 0.6511854748553768, 'train_rmse': 0.8069606402144883, 'test_rmse': 0.8853206526436642, 'test_mae': 0.6814780903817926}
{'epoch': 5, 'k': 5, 'train_mse': 0.5734973268724538, 'train_rmse': 0.7572960628924819, 'test_rmse': 0.8880621816933663, 'test_mae': 0.6830258665813669}
{'epoch': 6, 'k': 5, 'train_mse': 0.521595232002511, 'train_rmse': 0.7222155024662037, 'test_rmse': 0.8940854456868125, 'test_mae': 0.6873977555745747}
{'epoch': 7, 'k': 5, 'train_mse': 0.48785609

,k,RMSE_MF,MAE_MF,Precision@10,Recall@10,usuarios_avaliados_topk
0,5,0.913871,0.699990,0.021096,0.018393,602
1,10,0.933201,0.715435,0.018605,0.014258,602
2,20,0.952202,0.732796,0.028073,0.019252,602
3,50,0.959354,0.740427,0.040698,0.030915,602
4,100,0.916596,0.709279,0.055482,0.040693,602


Melhor k da Matrix Factorization por RMSE: 5


## 10. Comparação final dos três modelos

Agora vamos comparar:

1. Popularidade;
2. Melhor SVD encontrado;
3. Melhor Matrix Factorization encontrada.

A comparação será feita com:

- RMSE;
- MAE;
- Precision@10;
- Recall@10.

In [10]:
# Melhor SVD.
best_svd_scores = best_svd["scores"]
best_svd_k = best_svd["k"]
svd_test_pred = best_svd_scores[test_df["user_idx"].to_numpy(), test_df["movie_idx"].to_numpy()]
svd_rating_metrics = rating_metrics_from_predictions(
    f"SVD k={best_svd_k}",
    test_df["rating"].to_numpy(),
    svd_test_pred,
)
svd_topk_metrics = precision_recall_at_k(best_svd_scores, k=10)

# Melhor MF.
best_mf_scores = best_mf["scores"]
best_mf_k = best_mf["k"]
mf_test_pred = best_mf_scores[test_df["user_idx"].to_numpy(), test_df["movie_idx"].to_numpy()]
mf_rating_metrics = rating_metrics_from_predictions(
    f"Matrix Factorization k={best_mf_k}",
    test_df["rating"].to_numpy(),
    mf_test_pred,
)
mf_topk_metrics = precision_recall_at_k(best_mf_scores, k=10)

rating_results = pd.DataFrame([
    pop_rating_metrics,
    svd_rating_metrics,
    mf_rating_metrics,
])

top10_results = pd.DataFrame([
    {"modelo": "Popularidade", **pop_topk_metrics},
    {"modelo": f"SVD k={best_svd_k}", **svd_topk_metrics},
    {"modelo": f"Matrix Factorization k={best_mf_k}", **mf_topk_metrics},
])

print("Resultados de erro de nota:")
display(rating_results)

print("Resultados Top-10:")
display(top10_results)

rating_results.to_csv(RESULTS_DIR / "metrics_rating.csv", index=False)
top10_results.to_csv(RESULTS_DIR / "metrics_top10.csv", index=False)

# Resultado combinado do experimento de k.
experiment_k = svd_k_results.merge(mf_k_results, on="k", how="outer")
experiment_k.to_csv(RESULTS_DIR / "experiment_k.csv", index=False)
display(experiment_k)

Resultados de erro de nota:


,modelo,RMSE,MAE
0,Popularidade,0.966686,0.752467
1,SVD k=10,0.914034,0.707586
2,Matrix Factorization k=5,0.913871,0.699990


Resultados Top-10:


,modelo,Precision@10,Recall@10,usuarios_avaliados_topk
0,Popularidade,0.077907,0.061370,602
1,SVD k=10,0.127076,0.097155,602
2,Matrix Factorization k=5,0.021096,0.018393,602


,k,RMSE_SVD,MAE_SVD,Precision@10_x,Recall@10_x,usuarios_avaliados_topk_x,RMSE_MF,MAE_MF,Precision@10_y,Recall@10_y,usuarios_avaliados_topk_y
0,5,0.915704,0.708915,0.119601,0.090021,602,0.913871,0.699990,0.021096,0.018393,602
1,10,0.914034,0.707586,0.127076,0.097155,602,0.933201,0.715435,0.018605,0.014258,602
2,20,0.917398,0.709869,0.120764,0.095579,602,0.952202,0.732796,0.028073,0.019252,602
3,50,0.925810,0.717297,0.110631,0.107685,602,0.959354,0.740427,0.040698,0.030915,602
4,100,0.934168,0.726071,0.081728,0.088550,602,0.916596,0.709279,0.055482,0.040693,602


## 11. Exemplo de recomendação para um usuário

Vamos escolher automaticamente um usuário que tenha pelo menos alguns filmes relevantes no teste.

Depois vamos mostrar:

- filmes que ele avaliou bem no treino;
- recomendações por Popularidade;
- recomendações por SVD;
- recomendações por Matrix Factorization.

In [11]:
def movie_title_from_idx(movie_idx):
    movie_id = idx2movie[int(movie_idx)]
    if movie_id in movie_info.index:
        return movie_info.loc[movie_id, "title"]
    return f"movieId={movie_id}"


def recommend_for_user(user_idx, score_matrix, top_n=10):
    scores = np.array(score_matrix[user_idx], copy=True)
    seen_items = train_seen.get(user_idx, set())
    if seen_items:
        scores[list(seen_items)] = -np.inf

    top = np.argpartition(-scores, kth=min(top_n, len(scores) - 1))[:top_n]
    top = top[np.argsort(-scores[top])]

    rows = []
    for rank, movie_idx in enumerate(top, start=1):
        movie_id = idx2movie[int(movie_idx)]
        rows.append({
            "posicao": rank,
            "movieId": movie_id,
            "title": movie_title_from_idx(movie_idx),
            "score_previsto": float(scores[movie_idx]),
        })
    return pd.DataFrame(rows)

# Escolher usuário com mais filmes relevantes no teste.
if test_relevant:
    chosen_user_idx = max(test_relevant.keys(), key=lambda u: len(test_relevant[u]))
else:
    chosen_user_idx = int(test_df["user_idx"].iloc[0])

chosen_user_id = idx2user[chosen_user_idx]
print("Usuário escolhido:", chosen_user_id, "user_idx:", chosen_user_idx)

liked_train = train_df[(train_df["user_idx"] == chosen_user_idx) & (train_df["rating"] >= 4.0)].copy()
liked_train["title"] = liked_train["movie_idx"].apply(movie_title_from_idx)
liked_train = liked_train.sort_values("rating", ascending=False)[["movieId", "title", "rating"]].head(10)

print("Filmes que o usuário avaliou bem no treino:")
display(liked_train)

rec_pop = recommend_for_user(chosen_user_idx, pop_score_matrix, top_n=10)
rec_svd = recommend_for_user(chosen_user_idx, best_svd_scores, top_n=10)
rec_mf = recommend_for_user(chosen_user_idx, best_mf_scores, top_n=10)

print("Recomendações — Popularidade")
display(rec_pop)

print("Recomendações — SVD")
display(rec_svd)

print("Recomendações — Matrix Factorization")
display(rec_mf)

liked_train.to_csv(RESULTS_DIR / "example_user_liked_train.csv", index=False)
rec_pop.to_csv(RESULTS_DIR / "recommendations_popularity.csv", index=False)
rec_svd.to_csv(RESULTS_DIR / "recommendations_svd.csv", index=False)
rec_mf.to_csv(RESULTS_DIR / "recommendations_mf.csv", index=False)

recommendation_examples = pd.concat([
    rec_pop.assign(modelo="Popularidade"),
    rec_svd.assign(modelo="SVD"),
    rec_mf.assign(modelo="Matrix Factorization"),
], ignore_index=True)
recommendation_examples.to_csv(RESULTS_DIR / "recommendation_examples.csv", index=False)

Usuário escolhido: 414 user_idx: 413
Filmes que o usuário avaliou bem no treino:


,movieId,title,rating
50218,1277,Cyrano de Bergerac (1990),5.0
50220,1282,Fantasia (1940),5.0
50201,1240,"Terminator, The (1984)",5.0
50202,1247,"Graduate, The (1967)",5.0
50203,1249,"Femme Nikita, La (Nikita) (1990)",5.0
50204,1250,"Bridge on the River Kwai, The (1957)",5.0
50205,1252,Chinatown (1974),5.0
50206,1257,Better Off Dead... (1985),5.0
50207,1259,Stand by Me (1986),5.0
51850,79132,Inception (2010),5.0


Recomendações — Popularidade


,posicao,movieId,title,score_previsto
0,1,50,"Usual Suspects, The (1995)",4.218068
1,2,1221,"Godfather: Part II, The (1974)",4.186730
2,3,1104,"Streetcar Named Desire, A (1951)",4.156352
3,4,4011,Snatch (2000),4.135696
4,5,1198,Raiders of the Lost Ark (Indiana Jones and the...,4.121504
5,6,58559,"Dark Knight, The (2008)",4.119319
6,7,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),4.115263
7,8,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.110444
8,9,2324,Life Is Beautiful (La Vita è bella) (1997),4.095370
9,10,1136,Monty Python and the Holy Grail (1975),4.084321


Recomendações — SVD


,posicao,movieId,title,score_previsto
0,1,2797,Big (1988),3.604979
1,2,3753,"Patriot, The (2000)",3.579911
2,3,4979,"Royal Tenenbaums, The (2001)",3.573596
3,4,2953,Home Alone 2: Lost in New York (1992),3.568423
4,5,1760,Spice World (1997),3.567895
5,6,8376,Napoleon Dynamite (2004),3.567291
6,7,1265,Groundhog Day (1993),3.564666
7,8,8644,"I, Robot (2004)",3.562083
8,9,54503,Superbad (2007),3.556452
9,10,34,Babe (1995),3.554423


Recomendações — Matrix Factorization


,posicao,movieId,title,score_previsto
0,1,3473,Jonah Who Will Be 25 in the Year 2000 (Jonas q...,5.0
1,2,1136,Monty Python and the Holy Grail (1975),5.0
2,3,26131,"Battle of Algiers, The (La battaglia di Algeri...",5.0
3,4,44555,"Lives of Others, The (Das leben der Anderen) (...",5.0
4,5,5747,Gallipoli (1981),5.0
5,6,50,"Usual Suspects, The (1995)",5.0
6,7,8874,Shaun of the Dead (2004),5.0
7,8,106100,Dallas Buyers Club (2013),5.0
8,9,1281,"Great Dictator, The (1940)",5.0
9,10,167064,I Am Not Your Negro (2017),5.0


## 12. Salvar resumo final

Esta célula salva um arquivo `summary.json` com os principais números do experimento.

O notebook `relatorio.ipynb` vai usar esses arquivos para montar as tabelas e ajudar na escrita final.

In [12]:
summary = {
    "dataset": {
        "ratings_path": str(RATINGS_PATH),
        "movies_path": str(MOVIES_PATH),
        "n_users": int(n_users),
        "n_movies": int(n_movies),
        "n_ratings": int(n_ratings),
        "train_size": int(len(train_df)),
        "test_size": int(len(test_df)),
        "global_mean_train": float(GLOBAL_MEAN),
    },
    "best_svd_k": int(best_svd_k),
    "best_mf_k": int(best_mf_k),
    "chosen_user_id": int(chosen_user_id),
    "files_generated": [
        "results/metrics_rating.csv",
        "results/metrics_top10.csv",
        "results/experiment_k.csv",
        "results/recommendation_examples.csv",
        "results/summary.json",
    ],
}

with open(RESULTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Treinamento finalizado.")

{
  "dataset": {
    "ratings_path": "/Users/eduardo/Desktop/Insper/projeto_recommendation_system/data/ml_data/ratings.csv",
    "movies_path": "/Users/eduardo/Desktop/Insper/projeto_recommendation_system/data/ml_data/movies.csv",
    "n_users": 610,
    "n_movies": 9724,
    "n_ratings": 100836,
    "train_size": 80672,
    "test_size": 20164,
    "global_mean_train": 3.503421261404205
  },
  "best_svd_k": 10,
  "best_mf_k": 5,
  "chosen_user_id": 414,
  "files_generated": [
    "results/metrics_rating.csv",
    "results/metrics_top10.csv",
    "results/experiment_k.csv",
    "results/recommendation_examples.csv",
    "results/summary.json"
  ]
}
Treinamento finalizado.


## 13. Espaços para imagens dos experimentos

Depois de rodar os experimentos, você pode criar imagens e salvar em:

```text
docs/images/
```

Sugestões de imagens para colocar no README e no relatório:

```markdown
![Distribuição das notas](../docs/images/distribuicao_notas.png)
![Comparação de RMSE](../docs/images/rmse_comparacao.png)
![Comparação Top-10](../docs/images/top10_comparacao.png)
![Efeito do valor de k](../docs/images/efeito_k.png)
![Exemplo de recomendação](../docs/images/exemplo_recomendacao.png)
```

Este notebook não depende dessas imagens para funcionar. Elas são apenas para deixar o relatório visualmente melhor.